In [ ]:
#  get the code (push local changes to GitHub first)
# delete any previous clone so every run starts from a fresh copy
%cd /content
!rm -rf /content/nepali-summarizer
!git clone https://github.com/tekraj/nepali-summarizer.git /content/nepali-summarizer
%cd /content/nepali-summarizer

In [3]:
# install dependencies from pyproject.toml (keeps Colab's CUDA torch)
!pip install -q -e .

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.6/120.6 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.4/27.4 MB 47.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 544.1/544.1 kB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 120.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.2/110.2 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 1

In [ ]:
# data/cleaned is committed to git, so the clone above already includes it
!ls /content/nepali-summarizer/data/cleaned | wc -l

In [ ]:
# mount Drive so summaries survive a disconnect, then run the summarizer
from google.colab import drive
drive.mount('/content/drive')

import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/nepali-summarizer")  # notebooks have no __file__
sys.path.insert(0, str(PROJECT_ROOT))

from scripts.generate_summaries import run_summarizer

CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"
SUMMARY_DIR = Path("/content/drive/MyDrive/nepali-summarizer/summary")
DEFAULT_MODEL = "GenzNepal/mt5-summarize-nepali"
DEFAULT_BATCH_SIZE = 16
DEFAULT_TOKENIZER_MAX_LEN = 4096
DEFAULT_GENERATE_MAX_LEN = 1024
DEFAULT_FORCE = False

run_summarizer(
    model = DEFAULT_MODEL,
    batch_size = DEFAULT_BATCH_SIZE,
    tokenizer_max_len = DEFAULT_TOKENIZER_MAX_LEN,
    generate_max_len = DEFAULT_GENERATE_MAX_LEN,
    cleaned_dir = CLEANED_DIR,
    summary_dir = SUMMARY_DIR,
    force = DEFAULT_FORCE,
)

In [ ]:
#  zip the summaries on Drive and download them to this computer
# Locally, unzip into data/summary:  unzip -n summary.zip -d data/summary
import shutil
from google.colab import files

ZIP_BASE = Path("/content/summary")  # make_archive adds the .zip extension
zip_path = shutil.make_archive(str(ZIP_BASE), "zip", root_dir=SUMMARY_DIR)
print(f"Zipped {len(list(SUMMARY_DIR.glob('*.txt')))} summaries -> {zip_path}")

shutil.copy(zip_path, SUMMARY_DIR.parent / "summary.zip")  # backup copy on Drive
files.download(zip_path)